In [1]:
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3 datasets evaluate

In [2]:
import json, os, re, shutil, gc, time
import numpy as np
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# ── A100 memory & performance flags ─────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"]  = "expandable_segments:True"  # prevent fragmentation
os.environ["TOKENIZERS_PARALLELISM"]   = "false"                      # avoid fork warning with dataloader workers

torch.set_float32_matmul_precision("high")   # use TF32 tensor cores on A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32        = True

print("torch        :", torch.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("VRAM         :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


torch        : 2.8.0+cu128
CUDA         : True
GPU          : NVIDIA A100-SXM4-80GB
VRAM         : 85.1 GB


In [3]:
# ════════════════════════════════════════════════════════════
#  CONFIG  —  only edit this cell
# ════════════════════════════════════════════════════════════
MODEL_NAME   = "google/flan-t5-xl"
MAX_INPUT    = 512
MAX_TARGET   = 384          # plan tokens — kept at 384 to avoid truncation

# ── Batch / LR (tuned for A100 80 GB) ───────────────────────
BATCH_SIZE   = 32
GRAD_ACCUM   = 2
LR           = 5e-4
WARMUP_RATIO = 0.1
EPOCHS       = 10
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Data paths ───────────────────────────────────────────────
TRAIN_PATH = "./train_final.json"
DEV_PATH   = "./dev_final.json"

# ── Output paths ─────────────────────────────────────────────
CKPT_DIR   = "./checkpoints"
LORA_SAVE  = "./lora_adapter"

# ── Resume settings ──────────────────────────────────────────
RESUME    = False
PREV_CKPT = "./checkpoints/checkpoint-XXXX"

os.makedirs(CKPT_DIR,  exist_ok=True)
os.makedirs(LORA_SAVE, exist_ok=True)
print("Resume:", RESUME)
if RESUME:
    print("Checkpoint:", PREV_CKPT)
    assert Path(PREV_CKPT).exists(), f"Checkpoint not found: {PREV_CKPT}"
    assert (Path(PREV_CKPT) / "optimizer.pt").exists(), "optimizer.pt missing"
    assert (Path(PREV_CKPT) / "trainer_state.json").exists(), "trainer_state.json missing"
    with open(Path(PREV_CKPT) / "trainer_state.json") as f:
        ts = json.load(f)
    print(f"  Resuming from epoch {ts['epoch']:.1f}, global step {ts['global_step']}")


Resume: False


In [4]:
# ── Load data ────────────────────────────────────────────────
def load_json_safe(path):
    """Load a JSON file. If corrupted/truncated, recover all
    complete records by scanning line by line."""
    import os
    path = str(path)
    file_size = os.path.getsize(path)
    print(f"  File: {path}  ({file_size/1e6:.1f} MB)")
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        print(f"  Loaded OK: {len(data)} records")
        return data
    except json.JSONDecodeError as e:
        print(f"  WARNING: JSON parse failed ({e}). Attempting recovery...")
    records = []
    with open(path, encoding="utf-8", errors="replace") as f:
        raw = f.read()
    depth = 0; start = None; in_string = False; escape = False
    for i, ch in enumerate(raw):
        if escape: escape = False; continue
        if ch == "\\" and in_string: escape = True; continue
        if ch == '"' and not escape: in_string = not in_string
        if in_string: continue
        if ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                try: records.append(json.loads(raw[start:i+1]))
                except json.JSONDecodeError: pass
                start = None
    print(f"  Recovered {len(records)} complete records")
    if len(records) == 0:
        raise RuntimeError(f"Could not recover any records from {path}.")
    return records

print("Loading train...")
train_data = load_json_safe(TRAIN_PATH)
print("Loading dev...")
dev_data   = load_json_safe(DEV_PATH)

print(f"\nTrain : {len(train_data)} examples")
print(f"Dev   : {len(dev_data)} examples")
print()
sample = train_data[5]
print("INPUT :", sample["input"])
print("TARGET (plan only):", sample["output"])


Loading train...
  File: ./train_final.json  (6.3 MB)
  Loaded OK: 8659 records
Loading dev...
  File: ./dev_final.json  (0.7 MB)
  Loaded OK: 1034 records

Train : 8659 examples
Dev   : 1034 examples

INPUT : question: What are the names of the heads who are born outside the California state? | schema: head ( head_ID [PK], name, born_state ) | foreign keys: none
TARGET (plan only): step1: SCAN | table: head || step2: FILTER | condition: born_state <> 'California' || step3: PROJECT | columns: name


In [5]:
# ── Tokenizer ────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    # target = plan only (no SQL)
    inputs  = tokenizer(examples["input"],  max_length=MAX_INPUT,  truncation=True, padding=False)
    targets = tokenizer(examples["output"], max_length=MAX_TARGET, truncation=True, padding=False)
    inputs["labels"] = targets["input_ids"]
    return inputs

train_tok = Dataset.from_list([
    {"input": d["input"], "output": d["output"]} for d in train_data
]).map(preprocess, batched=True, remove_columns=["input", "output"])

dev_tok = Dataset.from_list([
    {"input": d["input"], "output": d["output"]} for d in dev_data
]).map(preprocess, batched=True, remove_columns=["input", "output"])

print("Train tokenized:", len(train_tok))
print("Dev   tokenized:", len(dev_tok))

lengths = [len(x["labels"]) for x in train_tok]
print(f"Target length — mean: {np.mean(lengths):.0f}  max: {np.max(lengths)}  p95: {np.percentile(lengths, 95):.0f}")
print("(If max == MAX_TARGET many examples are truncated — increase MAX_TARGET)")


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8659 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Train tokenized: 8659
Dev   tokenized: 1034
Target length — mean: 121  max: 384  p95: 284
(If max == MAX_TARGET many examples are truncated — increase MAX_TARGET)


In [6]:
# ── Model + LoRA ─────────────────────────────────────────────
base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.bfloat16,
    device_map  = "auto"
)

if RESUME:
    model = PeftModel.from_pretrained(base_model, PREV_CKPT, is_trainable=True)
    print("Loaded LoRA weights from:", PREV_CKPT)
else:
    lora_cfg = LoraConfig(
        task_type      = TaskType.SEQ_2_SEQ_LM,
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        target_modules = ["q", "k", "v", "o"],
        bias           = "none",
    )
    model = get_peft_model(base_model, lora_cfg)
    print("Fresh LoRA adapter initialised")

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
# Silence the HF warning: we manually manage use_cache via callback below.
# GC requires use_cache=False during forward pass — but eval/generate needs
# use_cache=True for KV-cache (otherwise generation is ~5x slower).
model.config.use_cache = False   # off for training (GC incompatible)
model.print_trainable_parameters()


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Fresh LoRA adapter initialised
trainable params: 18,874,368 || all params: 2,868,631,552 || trainable%: 0.6579572056523207


In [7]:
# No evaluation — CacheToggleCallback and compute_metrics are not needed.
# (kept as empty cell for reference)


In [8]:
# ── Training args ────────────────────────────────────────────
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

args = Seq2SeqTrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = EPOCHS,

    # ── Batch sizes ──────────────────────────────────────────
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,

    # ── Optimiser ────────────────────────────────────────────
    learning_rate               = LR,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = "cosine",
    weight_decay                = 0.01,

    # ── Precision ────────────────────────────────────────────
    bf16                        = True,
    tf32                        = True,

    # ── No evaluation ────────────────────────────────────────
    evaluation_strategy         = "no",
    predict_with_generate       = False,

    # ── Checkpointing / logging ───────────────────────────────
    save_strategy               = "epoch",
    save_total_limit            = 2,
    logging_steps               = 10,   # print train loss every 10 steps
    load_best_model_at_end      = False,

    # ── Dataloader ───────────────────────────────────────────
    group_by_length             = True,
    dataloader_num_workers      = 4,
    dataloader_pin_memory       = True,

    report_to                   = "none",
)

trainer = Seq2SeqTrainer(
    model         = model,
    args          = args,
    train_dataset = train_tok,
    tokenizer     = tokenizer,
    data_collator = data_collator,
    # No eval_dataset, no compute_metrics, no EarlyStoppingCallback
)


In [9]:
# ── Train ────────────────────────────────────────────────────
# Full memory flush before starting
torch.cuda.empty_cache()
gc.collect()
torch.cuda.synchronize()

# Print available VRAM before training starts (sanity check)
if torch.cuda.is_available():
    free = torch.cuda.mem_get_info()[0] / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM before train: {free:.1f} GB free / {total:.1f} GB total")

start = time.time()

if RESUME:
    trainer.train(resume_from_checkpoint=PREV_CKPT)
else:
    trainer.train()

elapsed = (time.time() - start) / 3600
print(f"\nDone in {elapsed:.2f} hours")


VRAM before train: 78.9 GB free / 85.1 GB total


Step,Training Loss
10,2.261800
20,2.699500
30,2.970200
40,1.897800
50,1.486300
60,1.170900
70,0.795400
80,0.580900
90,0.466600
100,0.319000


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/hug


Done in 1.02 hours


In [10]:
# ── Checkpoint listing ───────────────────────────────────────
print("\nAll checkpoint folders:")
for ckpt in sorted(Path(CKPT_DIR).glob("checkpoint-*")):
    print(f"  {ckpt.name}")
    for fname in sorted(f.name for f in ckpt.iterdir()):
        size = (ckpt / fname).stat().st_size
        print(f"      {fname:<40}  {size/1e6:.1f} MB")


All checkpoint folders:
  checkpoint-1219
      README.md                                 0.0 MB
      adapter_config.json                       0.0 MB
      adapter_model.safetensors                 37.8 MB
      optimizer.pt                              76.0 MB
      rng_state.pth                             0.0 MB
      scheduler.pt                              0.0 MB
      special_tokens_map.json                   0.0 MB
      tokenizer.json                            2.4 MB
      tokenizer_config.json                     0.0 MB
      trainer_state.json                        0.0 MB
      training_args.bin                         0.0 MB
  checkpoint-1350
      README.md                                 0.0 MB
      adapter_config.json                       0.0 MB
      adapter_model.safetensors                 37.8 MB
      optimizer.pt                              76.0 MB
      rng_state.pth                             0.0 MB
      scheduler.pt                              0.0 MB


In [11]:
# Evaluation disabled — training is input → plan only, no eval loop.
# If you want to check plan quality manually, use the inference cell below.
print("Training complete. No evaluation was run (evaluation_strategy='no').")


Training complete. No evaluation was run (evaluation_strategy='no').


In [12]:
# ── Save final LoRA adapter ───────────────────────────────────
model.save_pretrained(LORA_SAVE)
tokenizer.save_pretrained(LORA_SAVE)
print("LoRA adapter saved to:", LORA_SAVE)
for f in sorted(Path(LORA_SAVE).iterdir()):
    print(f"  {f.name:<40}  {f.stat().st_size/1e6:.2f} MB")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


LoRA adapter saved to: ./lora_adapter
  README.md                                 0.01 MB
  adapter_config.json                       0.00 MB
  adapter_model.safetensors                 37.83 MB
  special_tokens_map.json                   0.00 MB
  tokenizer.json                            2.42 MB
  tokenizer_config.json                     0.02 MB
